# v06 · Vocabulary factorization: reported reduction vs. what has to be stored

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
mf = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster_manifold.py", "mf")
V = mf.VocabFactorizer()
print("     L  reported  naive_bits  token_bits  +vocab_bits  +punct_bits  total   real_ratio")
for L in (47, 1000, 4000):
    t = CORPUS[:L]; va = V.encode(t); assert V.decode(va) == t
    r = va.bit_report(t)
    vocab_bits = sum(len(w) + 1 for w in va.vocab) * math.log2(97)               # vocabulary listed as text, 97-symbol charset
    punct_bits = sum(len(s) * math.log2(97) + math.log2(max(va.word_count, 2)) for _, s in va.punct_mask)
    total = r["vocab_token_bits"] + vocab_bits + punct_bits
    print("%6d  %8s  %10d  %10d  %11.0f  %11.0f  %6.0f  %.2f" % (L, r["reduction_factor"], r["naive_char_bits"], r["vocab_token_bits"], vocab_bits, punct_bits, total, r["naive_char_bits"]/total))

     L  reported  naive_bits  token_bits  +vocab_bits  +punct_bits  total   real_ratio
    47     25.9x         311          12          297           58     367  0.85
  1000      8.3x        6600         799         4897         2369    8065  0.82
  4000      7.2x       26400        3668        13682         9833   27183  0.97


The reported factor counts only the token-sequence address. The vocabulary and the punctuation spans are needed to decode and are not in it. With them counted (vocabulary and punctuation costed as text over the 97-symbol charset: an estimate, stated as such) the stored size exceeds the naive size. The round trip is exact in every row.